# LangGraph 实现智能体

本 notebook 用 LangGraph 的 `StateGraph` **手动编排**一个智能体（Agent），不使用 `create_agent` 等高层封装，目的是理解智能体的内部工作流程。

## 智能体工作流程

1. **用户提问** → 进入判断节点（classify）
2. **判断节点**：LLM 决定是直接回答，还是需要调用工具（天气/搜索）
3. 如果调用工具 → 工具返回结果 → **补充判断节点**（check）：信息够不够？
4. 如果不够 → 回到判断节点，换一个工具补充
5. 如果够了 → **最终答案节点**（final）：LLM 组织中文回答 → 结束

## 图结构

```
START → classify（LLM 判断路由）
            ├─ direct   ───────────────────→ final → END
            ├─ weather → weather_node → check
            └─ search  → search_node  → check
        check（LLM 判断信息够不够）
            ├─ enough → final → END
            └─ more   → classify（回到判断，换工具补充）
```

- `classify`：路由节点，输出 `direct` / `weather` / `search`
- `weather` / `search`：工具节点，调用外部 API
- `check`：补充判断节点，决定是否需要再次调用工具
- `final`：最终答案节点，LLM 综合信息生成回答
- 防止无限循环：最多 2 轮工具调用

## 1. 准备环境

In [2]:
# 导入依赖
import re
import json
import requests
import torch
from typing import TypedDict
from transformers import AutoModelForCausalLM, AutoTokenizer
from langgraph.graph import StateGraph, START, END

# 加载本地 Qwen1.5-1.8B 模型（base 基座模型，用 few-shot 提示引导）
MODEL_PATH = r's:\PythonAILearningCourse\15-transformers-basics\models\Qwen1.5-1.8B'
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()
EOS_IDS = [tokenizer.eos_token_id, 151645]

print('模型加载完成，设备:', model.device)

def llm_generate(prompt, max_new_tokens=100):
    # 本地模型生成函数（贪心解码 + 重复惩罚）
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            eos_token_id=EOS_IDS,
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True)
    return text.strip()

模型加载完成，设备: cuda:0


## 2. 定义工具

In [2]:
# 工具1：天气查询（使用 wttr.in 免费天气服务）
# 原 05 notebook 使用 senvious.com 天气 API，该域名已失效，改用 wttr.in
def ask_weather(city):
    # 查询指定城市的实时天气
    try:
        resp = requests.get(f'https://wttr.in/{city}', params={'format': 'j1'}, timeout=10)
        data = resp.json()
        cur = data['current_condition'][0]
        desc = cur['weatherDesc'][0]['value']
        temp = cur['temp_C']
        humidity = cur['humidity']
        return f'{city} weather: {desc}, temp {temp}C, humidity {humidity}%'
    except Exception as e:
        return f'weather query failed: {e}'

# 测试天气工具
print(ask_weather('Shenzhen'))

Shenzhen weather: Patchy rain nearby, temp 28C, humidity 64%


In [2]:
# 工具2：本地搜索引擎（SearXNG，端口 6688）
# 需要先启动本地搜索引擎：访问 http://localhost:6688
SEARCH_URL = 'http://localhost:6688/search'

def ask_search(query):
    # 通过本地 SearXNG 搜索引擎查询
    try:
        resp = requests.get(SEARCH_URL, params={'q': query, 'format': 'json'}, timeout=10)
        data = resp.json()
        results = data.get('results', [])[:5]
        if not results:
            return f'搜索{query}无结果'
        lines = []
        for i, r in enumerate(results, 1):
            title = r.get('title', '')
            content = r.get('content', '')[:200]
            lines.append(f'{i}. {title}\n   {content}')
        return '\n'.join(lines)
    except Exception as e:
        return f'搜索失败（请确认 localhost:6688 已启动）: {e}'

# 测试搜索引擎（未启动时会显示错误提示）
print(ask_search('Python'))

1. Python
   Topics referred to by the same term


## 3. 定义状态

In [2]:
class AgentState(TypedDict):
    question: str        # 用户原始问题
    action: str          # classify 决定的动作: direct/weather/search
    city: str            # 天气查询城市
    query: str           # 搜索关键词
    tool_result: str     # 工具返回结果
    tool_history: list   # 工具调用历史（防止重复）
    round: int           # 循环轮次（防止无限循环）
    need_more: bool      # check 节点判断是否需要补充
    final_answer: str    # 最终答案

## 4. 定义节点

> **注意**：本地 Qwen1.5-1.8B 是 base 基座模型，不服从只输出JSON类指令。
> 采用 **few-shot 纯文本补全 + 关键词规则兜底** 实现路由分类。

In [2]:
def classify_node(state: AgentState) -> dict:
    # 判断节点：用 few-shot 引导 base 模型做路由分类
    question = state['question']

    # few-shot 示例：严格区块格式，便于解析
    prompt = f'''请分析用户问题，决定下一步动作，并提取参数。

问题：深圳今天天气怎么样？
动作：weather
城市：Shenzhen

问题：LangGraph是什么？
动作：search
关键词：LangGraph

问题：你好，介绍一下你自己
动作：direct

问题：北京气温多少度？
动作：weather
城市：Beijing

问题：Python 3.13有什么新特性？
动作：search
关键词：Python 3.13 新特性

问题：{question}
动作：'''

    text = llm_generate(prompt, max_new_tokens=30)

    # 解析模型输出
    action, city, query = 'direct', '', ''
    for line in text.split('\n'):
        line = line.strip()
        if line.startswith('动作'):
            a = line.split('：')[-1].split(':')[-1].strip().lower()
            if 'weather' in a or '天气' in a:
                action = 'weather'
            elif 'search' in a or '搜索' in a:
                action = 'search'
            else:
                action = 'direct'
        elif line.startswith('城市'):
            city = line.split('：')[-1].split(':')[-1].strip()
        elif line.startswith('关键词'):
            query = line.split('：')[-1].split(':')[-1].strip()

    # 关键词规则兜底（base 模型可能不服从格式）
    if action == 'direct':
        if any(k in question for k in ['天气', '气温', '温度', '下雨']):
            action = 'weather'
            if not city:
                for c, e in [('北京','Beijing'),('上海','Shanghai'),('深圳','Shenzhen'),
                             ('广州','Guangzhou'),('杭州','Hangzhou'),('成都','Chengdu')]:
                    if c in question:
                        city = e
                        break
                if not city:
                    city = 'Beijing'
        elif any(k in question for k in ['是什么', '什么是', '搜索', '最新', '新闻']):
            action = 'search'
            if not query:
                query = question

    print(f'  [classify] action={action}, city={city}, query={query}')
    return {'action': action, 'city': city, 'query': query}

In [2]:
def weather_node(state: AgentState) -> dict:
    # 天气查询节点：调用天气API
    city = state.get('city', 'Beijing')
    result = ask_weather(city)
    history = state.get('tool_history', []) + [f'weather({city})']
    print(f'  [weather] query {city}: {result}')
    return {'tool_result': result, 'tool_history': history}

def search_node(state: AgentState) -> dict:
    # 搜索节点：调用本地搜索引擎
    query = state.get('query', state['question'])
    result = ask_search(query)
    history = state.get('tool_history', []) + [f'search({query})']
    print(f'  [search] search {query}: {result[:80]}...')
    return {'tool_result': result, 'tool_history': history}

In [2]:
def check_node(state: AgentState) -> dict:
    # 补充判断节点：用 few-shot 引导模型判断信息是否足够
    question = state['question']
    tool_result = state.get('tool_result', '')
    round_num = state.get('round', 0) + 1

    # 防止无限循环：最多 2 轮工具调用
    if round_num >= 2:
        print(f'  [check] max rounds ({round_num}), output directly')
        return {'round': round_num, 'need_more': False}

    tool_result_short = tool_result[:300] if tool_result else 'none'

    prompt = f'''请判断工具返回的信息是否足够回答用户问题。

问题：深圳天气
工具返回：Shenzhen weather: Sunny, temp 28C, humidity 65%
判断：enough

问题：LangGraph是什么
工具返回：搜索失败
判断：more

问题：{question}
工具返回：{tool_result_short}
判断：'''

    text = llm_generate(prompt, max_new_tokens=10)
    need_more = 'more' in text.lower() or '不够' in text or '不足' in text
    print(f'  [check] round={round_num}, need_more={need_more}')
    return {'round': round_num, 'need_more': need_more}

In [2]:
def final_node(state: AgentState) -> dict:
    # 最终答案节点：用 one-shot 引导模型生成中文回答
    question = state['question']
    tool_result = state.get('tool_result', '')

    if tool_result:
        prompt = f'''请根据参考信息回答用户问题。

问题：深圳今天天气怎么样？
参考信息：Shenzhen weather: Sunny, temp 28C, humidity 65%
回答：深圳今天天气晴朗，气温28度，湿度65%，适合外出活动。

问题：{question}
参考信息：{tool_result[:400]}
回答：'''
    else:
        prompt = f'''请回答用户问题。

问题：你好
回答：你好！我是智能助手，有什么可以帮助你的吗？

问题：{question}
回答：'''

    text = llm_generate(prompt, max_new_tokens=200)
    print(f'  [final] generating answer')
    return {'final_answer': text}

## 5. 构建图

In [2]:
def route_classify(state: AgentState) -> str:
    # classify 节点的路由函数：根据 action 决定下一步
    return state.get('action', 'direct')

def route_check(state: AgentState) -> str:
    # check 节点的路由函数：根据 need_more 决定下一步
    if state.get('round', 0) >= 2:
        return 'enough'
    return 'more' if state.get('need_more', False) else 'enough'

# 构建 StateGraph
graph = StateGraph(AgentState)

# 添加节点
graph.add_node('classify', classify_node)
graph.add_node('weather', weather_node)
graph.add_node('search', search_node)
graph.add_node('check', check_node)
graph.add_node('final', final_node)

# 添加边
graph.add_edge(START, 'classify')
graph.add_conditional_edges('classify', route_classify, {
    'direct': 'final',
    'weather': 'weather',
    'search': 'search',
})
graph.add_edge('weather', 'check')
graph.add_edge('search', 'check')
graph.add_conditional_edges('check', route_check, {
    'enough': 'final',
    'more': 'classify',
})
graph.add_edge('final', END)

# 编译
agent = graph.compile()
print('图构建完成！')

图构建完成！


## 6. 测试

In [2]:
# 测试1：天气问题（调用天气工具）
print('=' * 60)
print('测试1：深圳今天天气怎么样？')
print('=' * 60)
result = agent.invoke({
    'question': '深圳今天天气怎么样？',
    'action': '',
    'city': '',
    'query': '',
    'tool_result': '',
    'tool_history': [],
    'round': 0,
    'need_more': False,
    'final_answer': '',
})
print()
print('最终答案:')
print(result['final_answer'])

测试1：深圳今天天气怎么样？
  [classify] action=weather, city=Shenzhen, query=LangGraph
  [weather] query Shenzhen: Shenzhen weather: Patchy rain nearby, temp 28C, humidity 64%
  [check] round=1, need_more=False
  [final] generating answer

最终答案:
深圳今天天气多云转阴，气温28度，湿度64%。


In [1]:
# 测试2：搜索问题（调用搜索工具）
print('=' * 60)
print('测试2：LangGraph是什么？')
print('=' * 60)
result = agent.invoke({
    'question': 'LangGraph是什么？',
    'action': '',
    'city': '',
    'query': '',
    'tool_result': '',
    'tool_history': [],
    'round': 0,
    'need_more': False,
    'final_answer': '',
})
print()
print('最终答案:')
print(result['final_answer'])

测试2：LangGraph是什么？
  [classify] action=search, city=, query=LangGraph
  [search] search LangGraph: 搜索LangGraph无结果...
  [check] round=1, need_more=False
  [final] generating answer

最终答案:
LangGraph是谷歌公司开发的用于图数据库的开源软件。


In [1]:
# 测试3：闲聊（直接回答，不调用工具）
print('=' * 60)
print('测试3：你好，请介绍一下你自己')
print('=' * 60)
result = agent.invoke({
    'question': '你好，请介绍一下你自己',
    'action': '',
    'city': '',
    'query': '',
    'tool_result': '',
    'tool_history': [],
    'round': 0,
    'need_more': False,
    'final_answer': '',
})
print()
print('最终答案:')
print(result['final_answer'])

测试3：你好，请介绍一下你自己
  [classify] action=direct, city=, query=
  [final] generating answer

最终答案:
您好！我是一个人工智能助手，可以为您提供各种服务和帮助。请问您需要什么方面的帮助呢？


## 总结

### 为什么用本地 Qwen1.5-1.8B 而非商汤 SenseNova？
- 原 05 notebook 使用的商汤 SenseNova API（token.senvious.cn）和天气 API（api.senvious.com）域名已失效
- 改用本地 Qwen1.5-1.8B base 模型 + wttr.in 免费天气服务

### base 模型的适配策略
- base 基座模型不服从只输出JSON类指令
- 采用 **few-shot 纯文本补全** 引导模型输出结构化文本（动作/城市/关键词）
- **关键词规则兜底**：当模型输出格式不规范时，用关键词匹配决定路由
- 生成解码使用贪心模式 + 重复惩罚，避免采样噪声

### 手动编排 vs create_agent
- `create_agent`：高层封装，自动处理工具调用循环，简单但黑盒
- `StateGraph` 手动编排：每个节点/边都可见可控，适合学习智能体工作原理
- 本 notebook 用手动编排实现了和 create_agent 类似的功能：判断 -> 工具 -> 检查 -> 输出